# 14-day cumulative number of COVID-19 cases per 100 000 for districts (Landkreise) in Germany

At the end of the page, we provide a detailed description of how the numbers are calculated.

### Compute data

In [1]:
import datetime as dt
import pandas as pd
pd.set_option("max_rows", None)
from oscovida import get_incidence_rates_germany

## Table for all districts (=Landkreise)

In [2]:
cases_incidence, deaths_incidence = get_incidence_rates_germany()

Please be patient - downloading data from https://www.arcgis.com/sharing/rest/content/items/f10774f1c63e40168479a1feb6c7ca74/data ...
Completed downloading 227974 rows in 4.9 seconds.


In [3]:
cases_incidence

,14-day-sum,population,14-day-incidence-rate
Landkreis,,,
LK Ahrweiler,17,129727,13.1
LK Aichach-Friedberg,32,133596,24.0
LK Alb-Donau-Kreis,56,196047,28.6
LK Altenburger Land,5,90118,5.5
LK Altenkirchen,16,128705,12.4
LK Altmarkkreis Salzwedel,6,83765,7.2
LK Altötting,20,111210,18.0
LK Alzey-Worms,20,129244,15.5
LK Amberg-Sulzbach,20,103109,19.4


## Table sorted by 14-day-incidence

In [4]:
cases_incidence.sort_values(by="14-day-incidence-rate", ascending=False)

,14-day-sum,population,14-day-incidence-rate
Landkreis,,,
SK Hamm,286,179111,159.7
SK Remscheid,122,110994,109.9
LK Cloppenburg,180,169348,106.3
SK Berlin Friedrichshain-Kreuzberg,274,289762,94.6
SK München,1377,1471508,93.6
SK Berlin Mitte,345,384172,89.8
LK Dingolfing-Landau,84,96217,87.3
SK Würzburg,109,127880,85.2
SK Gelsenkirchen,211,260654,81.0


## Tutorial: Detailed calculation for one district

In [5]:
from oscovida import fetch_data_germany, germany_get_population
import datetime

In [6]:
period = 14 # Set the period we compute the incidence rate over

In [7]:
germany = fetch_data_germany() # Downloads population data for Germany
germany = germany.rename(
    columns={"AnzahlFall": "cases", "AnzahlTodesfall": "deaths"}
) # Rename the columns to English
germany.tail() # Print example of the table format and data

,FID,IdBundesland,Bundesland,Landkreis,Altersgruppe,Geschlecht,cases,deaths,Meldedatum,IdLandkreis,Datenstand,NeuerFall,NeuerTodesfall,Refdatum,NeuGenesen,AnzahlGenesen,IstErkrankungsbeginn,Altersgruppe2
date,,,,,,,,,,,,,,,,,,
2020-06-09,39098719,16,Thüringen,LK Altenburger Land,A80+,M,1,0,2020/06/09 00:00:00,16077,"28.09.2020, 00:00 Uhr",0,-9,2020/05/19 00:00:00,0,1,1,Nicht übermittelt
2020-05-06,39098720,16,Thüringen,LK Altenburger Land,A80+,W,1,0,2020/05/06 00:00:00,16077,"28.09.2020, 00:00 Uhr",0,-9,2020/05/04 00:00:00,0,1,1,Nicht übermittelt
2020-05-11,39098721,16,Thüringen,LK Altenburger Land,A80+,W,1,0,2020/05/11 00:00:00,16077,"28.09.2020, 00:00 Uhr",0,-9,2020/05/01 00:00:00,0,1,1,Nicht übermittelt
2020-05-28,39098722,16,Thüringen,LK Altenburger Land,A80+,W,2,0,2020/05/28 00:00:00,16077,"28.09.2020, 00:00 Uhr",0,-9,2020/05/28 00:00:00,0,2,0,Nicht übermittelt
2020-06-09,39098723,16,Thüringen,LK Altenburger Land,A80+,W,1,0,2020/06/09 00:00:00,16077,"28.09.2020, 00:00 Uhr",0,-9,2020/06/09 00:00:00,0,1,0,Nicht übermittelt


In [8]:
yesterday = datetime.datetime.now() - datetime.timedelta(days=1)
fortnight_ago = yesterday - datetime.timedelta(days=period)
periods = (fortnight_ago < germany.index) & (germany.index < yesterday) # Create a period mask
germany = germany.iloc[periods]

In [9]:
cases_germany = germany[["Landkreis", "cases"]]
cases_germany.tail() # The data for germany has the date as the index, and the location data on each column

,Landkreis,cases
date,,
2020-09-24,LK Altenburger Land,1
2020-09-22,LK Altenburger Land,1
2020-09-15,LK Altenburger Land,1
2020-09-22,LK Altenburger Land,1
2020-09-22,LK Altenburger Land,1


In [10]:
cases_sum = (
    cases_germany.groupby("Landkreis")
    .sum()
    .rename(columns={"cases": f"{period}-day-sum"})
) # Here we sum the daily data to get total data
cases_sum.tail()

,14-day-sum
Landkreis,
SK Worms,36
SK Wuppertal,150
SK Würzburg,109
SK Zweibrücken,10
StadtRegion Aachen,182


In [11]:
population = (
    germany_get_population()
    .population
    .to_frame()
) # This gives us population per county, with county as the index
population.tail()

,population
county,
SK Berlin Spandau,243977
SK Berlin Steglitz-Zehlendorf,308697
SK Berlin Mitte,384172
SK Berlin Friedrichshain-Kreuzberg,289762
SK Berlin Tempelhof-Schöneberg,351644


In [12]:
cases_incidence = cases_sum.join(population)
cases_incidence[f"{period}-day-incidence-rate"] = (
    cases_incidence[f"{period}-day-sum"] / cases_incidence["population"] * 100_000
) # By convention the incedence rate is the total cases in the past x days / population * 100_000

In [13]:
cases_incidence.tail()

,14-day-sum,population,14-day-incidence-rate
Landkreis,,,
SK Worms,36,83330,43.201728
SK Wuppertal,150,354382,42.327206
SK Würzburg,109,127880,85.236159
SK Zweibrücken,10,34209,29.232073
StadtRegion Aachen,182,555465,32.765341


In [14]:
cases_incidence.loc['SK Hamburg']

14-day-sum               8.010000e+02
population               1.841179e+06
14-day-incidence-rate    4.350473e+01
Name: SK Hamburg, dtype: float64